# LLM-RecFusion — Stage 2: Baseline ALS Recall (Traditional Recaller)

**Objective**: Deploy a lightweight `implicit.ALS` (Alternating Least Squares) model as the **traditional recall safety net** — producing the baseline Recall@100 / NDCG@100 against which the upcoming LLM-enhanced dual-tower model will be compared.

**Why ALS for Implicit Feedback?**

| Aspect | ALS | Traditional SVD |
|--------|-----|-----------------|
| Loss function | Weighted squared error on **observed + unobserved** | Squared error on **observed only** |
| Confidence weighting | Explicit `confidence = 1 + alpha * rating` for unobserved entries | No confidence mechanism — treats missing as zero with equal weight |
| Optimization | Alternating least squares — fix user factors, solve item factors (and vice versa) | Gradient-based — requires learning rate tuning, slower convergence |
| Sparsity handling | Naturally handles implicit signals (clicks, views, purchases) | Degraded when interaction matrix is >99% sparse |

> **Why ALS > SVD for implicit feedback**: In implicit scenarios (clicks, views, purchase history), the absence of interaction is *weakly negative*, not zero. ALS weights unobserved entries down (via confidence `C=1` vs `C=1+alpha` for observed), while SVD treats missing as zero with equal weight — essentially saying "not watching = disliking," which is false in recommendation.

---
### Pipeline Position

```
 Stage 1                Stage 2 (This Notebook)          Stage 3-4
 +---------------+       +--------------------+        +------------------+
 | Data Proc.    | --->  | ALS Baseline       | --->   | LLM Dual-Tower   |
 | Feature Eng.  |       | Recall@100/NDCG@100|        | Multi-Route      |
 | Augmentation  |       | (Traditional Path) |        | Fusion           |
 +---------------+       +---------+----------+        +------------------+
                                   |
                                   v
                          baseline_recall_results.parquet
                          (For multi-route fusion in Task 4)
```

In [10]:
# ============================================================
# Cell 1: Imports & Global Configuration
# ============================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ---------- Paths ----------
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

print(f"✅ Notebook 当前运行目录: {Path.cwd()}")
print(f"✅ 项目根目录: {BASE_DIR}")
print(f"✅ 数据目录存在吗? {PROCESSED_DIR.exists()}")

# ---------- ALS Hyper-parameters ----------
ALS_FACTORS = 64          # Latent factor dimensionality
ALS_REGULARIZATION = 0.01 # L2 regularization strength
ALS_ITERATIONS = 15       # Number of alternating least squares iterations
ALS_ALPHA = 40.0          # Confidence scaling factor (C = 1 + alpha * value)
TOP_K = 100               # Number of candidates to recall per user
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print(f"ALS config: factors={ALS_FACTORS}, reg={ALS_REGULARIZATION}, "
      f"iter={ALS_ITERATIONS}, alpha={ALS_ALPHA}, top_k={TOP_K}")

✅ Notebook 当前运行目录: /home/hjy/Videos/LLM-REC/notebooks
✅ 项目根目录: /home/hjy/Videos/LLM-REC
✅ 数据目录存在吗? True
ALS config: factors=64, reg=0.01, iter=15, alpha=40.0, top_k=100


---
## 1. Data Loading & Interaction Matrix Construction

### Why CSR Matrix?

The `implicit` library requires inputs as `scipy.sparse.csr_matrix` (Compressed Sparse Row format) because:
- **Memory efficiency**: A 6,040 x 3,706 matrix with ~1M observed entries = ~99.9% sparsity. CSR stores only non-zero elements, using ~8MB instead of ~180MB for dense.
- **Fast row slicing**: User-side operations (get all items for user `u`) are O(1) — critical for ALS's alternating least squares step.
- **BLAS-friendly**: CSR's continuous row-major layout maps naturally to BLAS Level 2/3 operations.

### Matrix Construction Strategy

We build the user-item interaction matrix from all positive (label=1) training samples. Since the sequence-constructed training data no longer carries raw ratings, we use **binary implicit feedback** (value=1.0), which is the standard approach for production ALS in implicit feedback settings (clicks, views, purchases). The ALS model's `alpha` parameter internally scales confidence.

**Important API note**: `implicit.ALS.fit()` accepts a (users x items) CSR matrix directly — contrary to outdated documentation that claims it needs transposed input. The library internally handles the coordinate format.

In [4]:
# ============================================================
# Cell 3: Load Positive Interactions from Train Data
# ============================================================

# --- 3a. Load training positive samples (label == 1) ---
print("=" * 60)
print("【1/4】Loading training interactions and building CSR matrix...")
print("=" * 60)

train_df = pd.read_parquet(PROCESSED_DIR / "train_data.parquet")
train_pos = train_df[train_df["label"] == 1].copy()
print(f"  Training data (total):      {len(train_df):>10,}")
print(f"  Positive interactions only:  {len(train_pos):>10,}")
print(f"  Unique users:               {train_pos['user_id_encoded'].nunique():>10,}")
print(f"  Unique items:               {train_pos['target_movie_id'].nunique():>10,}")

# --- 3b. Get global ID mappings ---
# We map from 1-based encoded IDs to 0-based matrix indices
user_ids = train_pos["user_id_encoded"].unique()
item_ids = train_pos["target_movie_id"].unique()

user2idx = {uid: i for i, uid in enumerate(sorted(user_ids))}
item2idx = {iid: i for i, iid in enumerate(sorted(item_ids))}
idx2item = {i: iid for iid, i in item2idx.items()}

n_users = len(user_ids)
n_items = len(item_ids)
print(f"  Matrix shape:               {n_users} users x {n_items} items")
print(f"  Density:                    {len(train_pos) / (n_users * n_items):.6%}")

del train_df  # free memory

【1/4】Loading training interactions and building CSR matrix...
  Training data (total):       3,949,525
  Positive interactions only:     789,905
  Unique users:                    6,040
  Unique items:                    3,663
  Matrix shape:               6040 users x 3663 items
  Density:                    3.570270%


In [5]:
# ============================================================
# Cell 4: Build CSR Matrix (Binary Implicit Feedback)
# ============================================================

# Map user and item IDs to 0-based indices in one vectorized pass
rows = train_pos["user_id_encoded"].map(user2idx).values.astype(np.int32)
cols = train_pos["target_movie_id"].map(item2idx).values.astype(np.int32)

# --- Binary implicit feedback ---
# The sequence-constructed training data no longer carries raw ratings,
# so we use binary value 1.0 for all observed interactions.
# The ALS model's own alpha parameter handles confidence scaling:
#   confidence = 1 + alpha * value
# This gives observed entries weight = 1 + 40 = 41 vs unobserved weight = 1
data = np.ones(len(train_pos), dtype=np.float64)

print(f"  Using binary implicit feedback: {len(data):,} entries with value=1.0")
print(f"  Observed confidence weight:     {1 + ALS_ALPHA} (vs 1.0 for unobserved)")

# Build CSR matrix -- the core input for implicit ALS
# shape = (n_users, n_items), dtype = float64
interaction_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(n_users, n_items),
    dtype=np.float64,
)
print(f"  CSR matrix shape:           {interaction_matrix.shape}")
print(f"  Non-zero entries:           {interaction_matrix.nnz:,}")
print(f"  Sparsity:                   {1 - interaction_matrix.nnz / (n_users * n_items):.6%}")
print(f"  Memory:                     {interaction_matrix.data.nbytes / 1024**2:.2f} MB")

# Free large DataFrames
del train_pos, rows, cols, data

  Using binary implicit feedback: 789,905 entries with value=1.0
  Observed confidence weight:     41.0 (vs 1.0 for unobserved)
  CSR matrix shape:           (6040, 3663)
  Non-zero entries:           789,905
  Sparsity:                   96.429730%
  Memory:                     6.03 MB


---
## 2. ALS Model Training

### How ALS Works (Concise)

1. **Factorize**: Decompose the user-item interaction matrix `R =approx U * V^T`, where `U in R^{u x f}`, `V in R^{i x f}`.
2. **Alternate**: Fix `V`, solve for each user `u`:
   `U_u = (V^T * C_u * V + lambda*I)^{-1} * V^T * C_u * R_u`
   Then fix `U`, solve for each item `i` symmetrically.
3. **Converge**: Repeat until loss stabilizes or max iterations reached.

The `implicit` library uses a **conjugate gradient (CG) solver** under the hood, making each iteration extremely fast even for large matrices. The input matrix is expected as (users x items) in CSR format.

In [6]:
# ============================================================
# Cell 6: Fit AlternatingLeastSquares Model
# ============================================================
print("=" * 60)
print("【2/4】Training ALS model...")
print("=" * 60)

# Note: implicit.ALS.fit() accepts a (users x items) CSR matrix directly.
# Internally, it transposes for its native (items, users) coordinate format.
# user_factors shape = (n_users, factors), item_factors = (n_items, factors).
als_model = AlternatingLeastSquares(
    factors=ALS_FACTORS,
    regularization=ALS_REGULARIZATION,
    iterations=ALS_ITERATIONS,
    alpha=ALS_ALPHA,          # Confidence scaling for unobserved entries
    random_state=RANDOM_SEED,
    use_gpu=False,            # CPU is sufficient for 6k x 3.7k matrix
)

%time als_model.fit(interaction_matrix, show_progress=True)

print(f"\n  ALS training complete!")
print(f"  User factors shape:         {als_model.user_factors.shape}")
print(f"  Item factors shape:         {als_model.item_factors.shape}")

【2/4】Training ALS model...


100%|██████████| 15/15 [00:00<00:00, 26.58it/s]

CPU times: user 6.25 s, sys: 317 ms, total: 6.56 s
Wall time: 577 ms

  ALS training complete!
  User factors shape:         (6040, 64)
  Item factors shape:         (3663, 64)


---
## 3. Offline Evaluation: Recall@K & NDCG@K

### Metric Formulas

**Recall@K**:
$$Recall@K = (1 / |U_test|) * sum_{u in U_test} |TopK(u) intersect GT(u)| / |GT(u)|$$

> Measures the fraction of ground-truth items successfully retrieved in the top-K. Higher = better coverage of user's actual preferences.

**NDCG@K** (Normalized Discounted Cumulative Gain):
$$DCG@K = sum_{i=1}^{K} rel_i / log_2(i + 1),  NDCG@K = DCG@K / IDCG@K$$

> Measures ranking quality — higher-ranked relevant items contribute more. NDCG normalizes by the ideal ranking, so it's always between 0 and 1.

### Evaluation Protocol

For each user in the **test set**:
1. Retrieve Top-100 items from ALS (excluding items already interacted with in training via `filter_already_liked_items=True`).
2. Check which ground-truth items (from test set positive interactions) appear in the candidate list.
3. Compute per-user Recall@100 and NDCG@100, then macro-average across all users.

In [7]:
# ============================================================
# Cell 8: Prepare Ground Truth
# ============================================================
print("=" * 60)
print("【3/4】Preparing evaluation data...")
print("=" * 60)

# --- 8a. Load test positive interactions ---
print("  Loading test data...")
test_df = pd.read_parquet(PROCESSED_DIR / "test_data.parquet")
test_pos = test_df[test_df["label"] == 1].copy()
del test_df

print(f"  Test positive interactions: {len(test_pos):,}")
print(f"  Unique test users:          {test_pos['user_id_encoded'].nunique():,}")

# --- 8b. Build per-user ground truth ---
# Group ground-truth items by user for efficient evaluation
test_user_groups = test_pos.groupby("user_id_encoded")["target_movie_id"].apply(set).to_dict()
print(f"  Ground truth users:         {len(test_user_groups)}")

del test_pos

【3/4】Preparing evaluation data...
  Loading test data...
  Test positive interactions: 102,132
  Unique test users:          6,040
  Ground truth users:         6040


In [11]:
# ============================================================
# Cell 9: Evaluate Recall@K & NDCG@K
# ============================================================

def compute_recall_and_ndcg(
    model,
    interaction_matrix,
    test_user_groups,
    user2idx,
    idx2item,
    top_k=100,
):
    # Evaluate ALS model on test set.
    #
    # Protocol:
    #   1. For each test user, get Top-K recommendations from ALS,
    #      excluding items already interacted with in training.
    #   2. Compute Recall@K and NDCG@K per user.
    #   3. Macro-average across all test users.
    #
    # Parameters
    # ----------
    # model : AlternatingLeastSquares
    # interaction_matrix : csr_matrix, shape (n_users, n_items)
    # test_user_groups : dict {user_id_encoded: set(target_movie_ids)}
    # user2idx : dict {user_id_encoded: 0-based row index}
    # idx2item : dict {0-based column index: movie_id_encoded}
    # top_k : int
    #
    # Returns
    # -------
    # mean_recall, mean_ndcg, per_user_recalls, per_user_ndcgs
    recall_list = []
    ndcg_list = []

    valid_users = [
        uid for uid in test_user_groups
        if uid in user2idx and len(test_user_groups[uid]) > 0
    ]

    print(f"  Evaluating on {len(valid_users)} users...")

    for uid in tqdm(valid_users, desc="Evaluating"):
        user_idx = user2idx[uid]
        ground_truth = test_user_groups[uid]

        # Step 1: Get Top-K recommendations via ALS
        # recommend(user_index, user_row, N=top_k, filter_already_liked_items=True)
        # Returns (item_indices_in_model, scores)
        # User row is the row of interaction_matrix for this user,
        # used by implicit to know which items to filter.
        recommendations, scores = model.recommend(
            user_idx,
            interaction_matrix[user_idx],
            N=top_k,
            filter_already_liked_items=True,
        )

        # Map internal item indices to original movie_id_encoded
        rec_items = {idx2item[idx] for idx in recommendations}

        # Step 2: Compute Recall@K
        n_hits = len(rec_items & ground_truth)
        recall = n_hits / len(ground_truth) if len(ground_truth) > 0 else 0.0
        recall_list.append(recall)

        # Step 3: Compute NDCG@K
        # DCG = sum(rel_i / log2(i+1)) where rel_i = 1 if item in ground truth
        dcg = 0.0
        for rank, item_idx in enumerate(recommendations[:top_k]):
            if idx2item[item_idx] in ground_truth:
                dcg += 1.0 / np.log2(rank + 2)  # +2 because log2(1) = 0 -> inf

        # IDCG: ideal DCG -- all relevant items ranked at the top
        ideal_hits = min(len(ground_truth), top_k)
        idcg = sum(1.0 / np.log2(r + 2) for r in range(ideal_hits))

        ndcg = dcg / idcg if idcg > 0 else 0.0
        ndcg_list.append(ndcg)

    mean_recall = float(np.mean(recall_list))
    mean_ndcg = float(np.mean(ndcg_list))

    return mean_recall, mean_ndcg, recall_list, ndcg_list


# --- Run evaluation ---
print("=" * 60)
print("【3/4 continued】Running ALS evaluation...")
print("=" * 60)

mean_recall, mean_ndcg, per_user_recalls, per_user_ndcgs = compute_recall_and_ndcg(
    model=als_model,
    interaction_matrix=interaction_matrix,
    test_user_groups=test_user_groups,
    user2idx=user2idx,
    idx2item=idx2item,
    top_k=TOP_K,
)

print(f"\n{'=' * 60}")
print(f"📊 ALS Baseline Evaluation Results")
print(f"{'=' * 60}")
print(f"  Hyper-parameters:")
print(f"    factors          = {ALS_FACTORS}")
print(f"    regularization   = {ALS_REGULARIZATION}")
print(f"    iterations       = {ALS_ITERATIONS}")
print(f"    alpha            = {ALS_ALPHA}")
print(f"    top_k            = {TOP_K}")
print(f"  {'-' * 48}")
print(f"  🔹 Recall@{TOP_K} (mean) : {mean_recall:.4f} ({mean_recall*100:.2f}%)")
print(f"  🔹 NDCG@{TOP_K}  (mean) : {mean_ndcg:.4f}")
print(f"  {'-' * 48}")
print(f"  📈 Per-user Recall@{TOP_K} stats:")
print(f"      min    = {np.min(per_user_recalls):.4f}")
print(f"      median = {np.median(per_user_recalls):.4f}")
print(f"      max    = {np.max(per_user_recalls):.4f}")
print(f"      std    = {np.std(per_user_recalls):.4f}")
print(f"  📈 Per-user NDCG@{TOP_K} stats:")
print(f"      min    = {np.min(per_user_ndcgs):.4f}")
print(f"      median = {np.median(per_user_ndcgs):.4f}")
print(f"      max    = {np.max(per_user_ndcgs):.4f}")
print(f"      std    = {np.std(per_user_ndcgs):.4f}")
print(f"{'=' * 60}")

【3/4 continued】Running ALS evaluation...
  Evaluating on 6040 users...















Evaluating: 100%|██████████| 6040/6040 [00:01<00:00, 4811.82it/s]


📊 ALS Baseline Evaluation Results
  Hyper-parameters:
    factors          = 64
    regularization   = 0.01
    iterations       = 15
    alpha            = 40.0
    top_k            = 100
  ------------------------------------------------
  🔹 Recall@100 (mean) : 0.3014 (30.14%)
  🔹 NDCG@100  (mean) : 0.1350
  ------------------------------------------------
  📈 Per-user Recall@100 stats:
      min    = 0.0000
      median = 0.2273
      max    = 1.0000
      std    = 0.2700
  📈 Per-user NDCG@100 stats:
      min    = 0.0000
      median = 0.1089
      max    = 1.0000
      std    = 0.1208


---
## 4. Results Persistence (For Multi-Route Fusion)

### Why Save to Parquet?

In **Task 4 (Multi-Route Fusion)**, we will merge the ALS recall results with:
- LLM dual-tower recall results (semantic retrieval)
- Potentially other recall paths (ItemCF, Popularity, etc.)

Each recall path produces a DataFrame with the **same schema**:
- `user_id_encoded`: original encoded user ID
- `item_id_encoded`: original encoded movie ID
- `score`: ALS predicted relevance score (higher = more relevant)
- `rank`: position in the Top-100 list (1-based)
- `recall_source`: identifier string, e.g. `"ALS_Baseline"`

The fusion layer will read all parquet files, merge scores using a stacking/weighting strategy, and produce the final candidate list.

In [12]:
# ============================================================
# Cell 11: Persist Top-100 Recall Results for Multi-Route Fusion
# ============================================================
print("=" * 60)
print("【4/4】Saving Top-100 recall results for multi-route fusion...")
print("=" * 60)

# --- 11a. Generate recommendations for ALL test users ---
print("  Generating Top-100 recommendations for all test users...")

all_result_rows = []

for uid in tqdm(test_user_groups, desc="Generating Top-100"):
    if uid not in user2idx:
        continue  # safety skip for users only in test

    user_idx = user2idx[uid]

    rec_indices, rec_scores = als_model.recommend(
        user_idx,
        interaction_matrix[user_idx],
        N=TOP_K,
        filter_already_liked_items=True,
    )

    for rank, (item_idx, score) in enumerate(zip(rec_indices, rec_scores)):
        all_result_rows.append({
            "user_id_encoded": uid,
            "item_id_encoded": idx2item[item_idx],
            "score": float(score),
            "rank": rank + 1,  # 1-based rank
            "recall_source": "ALS_Baseline",
        })

# --- 11b. Build DataFrame ---
result_df = pd.DataFrame(all_result_rows)
result_df["user_id_encoded"] = result_df["user_id_encoded"].astype(np.int32)
result_df["item_id_encoded"] = result_df["item_id_encoded"].astype(np.int32)
result_df["rank"] = result_df["rank"].astype(np.int16)
result_df["score"] = result_df["score"].astype(np.float32)

print(f"  Result shape:               {result_df.shape}")
print(f"  Unique users:               {result_df['user_id_encoded'].nunique()}")
print(f"  Total recommendations:      {len(result_df):,}")
print(f"  (= {len(test_user_groups)} users x {TOP_K} items)")

# --- 11c. Preview ---
print(f"\n  Preview (first 10 rows):")
display(result_df.head(10))

# --- 11d. Save to Parquet ---
output_path = PROCESSED_DIR / "baseline_recall_results.parquet"
result_df.to_parquet(output_path, index=False)
file_size_mb = output_path.stat().st_size / 1024 / 1024
print(f"\n  Saved to: {output_path}")
print(f"  File size:                 {file_size_mb:.2f} MB")

print(f"\n{'=' * 60}")
print(f"✅ Stage 2 — Baseline ALS Recall Complete!")
print(f"{'=' * 60}")
print()
print("  What's next:")
print("  +" + "-" * 62 + "+")
print("  | Task 2: LLM Dual-Tower Recall (semantic retrieval)          |")
print("  |   - Generate item embeddings using LLM (text to vector)     |")
print("  |   - Build user towers from interaction history              |")
print("  |   - Compare metrics against ALS baseline                    |")
print("  |                                                             |")
print("  | Task 3-4: Multi-Route Fusion                                |")
print(f"  |   - Load {output_path.name} from disk                      |")
print("  |   - Merge with LLM recall results                           |")
print("  |   - Apply stacking / weighted fusion                        |")
print("  |   - Evaluate fused recall metrics                           |")
print("  +" + "-" * 62 + "+")

【4/4】Saving Top-100 recall results for multi-route fusion...
  Generating Top-100 recommendations for all test users...



Exception ignored in: <function tqdm.__del__ at 0x7961450b5480>
Traceback (most recent call last):
  File "/home/hjy/miniconda3/envs/llmrec/lib/python3.10/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/hjy/miniconda3/envs/llmrec/lib/python3.10/site-packages/tqdm/notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'













Generating Top-100: 100%|██████████| 6040/6040 [00:01<00:00, 4263.78it/s]


  Result shape:               (604000, 5)
  Unique users:               6040
  Total recommendations:      604,000
  (= 6040 users x 100 items)

  Preview (first 10 rows):


,user_id_encoded,item_id_encoded,score,rank,recall_source
0,1,610,1.058137,1,ALS_Baseline
1,1,11,1.057247,2,ALS_Baseline
2,1,597,1.049801,3,ALS_Baseline
3,1,65,1.043757,4,ALS_Baseline
4,1,694,1.036585,5,ALS_Baseline
5,1,105,1.015339,6,ALS_Baseline
6,1,391,1.004933,7,ALS_Baseline
7,1,208,0.996873,8,ALS_Baseline
8,1,125,0.987151,9,ALS_Baseline
9,1,128,0.986647,10,ALS_Baseline



  Saved to: /home/hjy/Videos/LLM-REC/data/processed/baseline_recall_results.parquet
  File size:                 3.81 MB

✅ Stage 2 — Baseline ALS Recall Complete!

  What's next:
  +--------------------------------------------------------------+
  | Task 2: LLM Dual-Tower Recall (semantic retrieval)          |
  |   - Generate item embeddings using LLM (text to vector)     |
  |   - Build user towers from interaction history              |
  |   - Compare metrics against ALS baseline                    |
  |                                                             |
  | Task 3-4: Multi-Route Fusion                                |
  |   - Load baseline_recall_results.parquet from disk                      |
  |   - Merge with LLM recall results                           |
  |   - Apply stacking / weighted fusion                        |
  |   - Evaluate fused recall metrics                           |
  +--------------------------------------------------------------+
